# Entity ***Mints***
+ Layer **bronze**
+ Priority **2**
---
### Imports

In [0]:
%run ../common/bronze_constants

In [0]:
%run ../common/medallion_functions

In [0]:
import requests
import time
from datetime import datetime, timedelta, timezone
from pyspark.sql.functions import current_timestamp

### Parameters

In [0]:
layer = dbutils.widgets.get("layer")
entity_name = dbutils.widgets.get("entity_name")
load_pattern = dbutils.widgets.get("load_pattern")

### Subgraph API

In [0]:
SUBGRAPH_URL = get_subgraph_api_url_from_secrets()

### Variables

In [0]:
days = 4
mints_list = []
query_variables = {
    "batch_pool_ids": [],
    "skip": 0,
    "since_ts": int((datetime.now(timezone.utc)- timedelta(days=days)).timestamp())
}

In [0]:
mints_query = """
  query MintsQuery($batch_pool_ids: [String!]!, $skip: Int!, $since_ts: BigInt!){
    mints(
      where: {
        pool_in: $batch_pool_ids,
        timestamp_gte: $since_ts
      },
      first: 1000,
      skip: $skip
    ) {
      id
      timestamp
      pool {
        id
      }
      token0 {
        id
      }
      token1 {
        id
      }
      transaction {
        id
      }
      tickLower
      tickUpper
      owner
      sender
      origin
      amount
      amount0
      amount1
      amountUSD
      logIndex
    }
  }
  """

### Filtering relevant pools
+ Total value locked > $1.000
+ Volume > $500
+ Transaction count > 5


In [0]:
filtered_bz_pools_df = retrieve_deduped_relevant_bronze_pools(
    tvl_usd = 1000,
    vol_usd = 500,
    tx_count = 5
    )

### Execution

In [0]:
pool_ids_list = [row.id for row in filtered_bz_pools_df.select("id").collect()]

In [0]:
loaded_pools_count = 0
batch = 1

for i in range(0, len(pool_ids_list), BATCH_SIZE):
    batch_pools = pool_ids_list[i:i+BATCH_SIZE]
    query_variables["batch_pool_ids"] = batch_pools
    query_variables["skip"] = 0
    page = 1
    print(f"*INFO*: Loading mints for {len(batch_pools)} pools.")

    while True:
        start_time = time.time()

        try:
            response = requests.post(SUBGRAPH_URL, json={"query": mints_query, "variables": query_variables})
            
            if response.status_code == 200:
                mints_data = response.json()["data"][entity_name]
            else:
                print(f"*ERROR*:{response.status_code}")
                mints_data = None

        except Exception as err:
            print(f"*ERROR*: Request failed with error: {err}")
            mints_data = None
        
        mints_list.extend(mints_data)
        time.sleep(0.5)
        rows_loaded = len(mints_data)
        end_time = time.time() - start_time
        f_end_time = str(timedelta(seconds=int(end_time)))
        print(f"*INFO*: Batch: {batch}, Page: {page}, Total time: {f_end_time}, Loaded rows: {rows_loaded}, Skip size: {query_variables["skip"]}")
        page += 1
        query_variables["skip"] += PAGE_SIZE
        
        if rows_loaded < PAGE_SIZE:
            break
    
    loaded_pools_count += len(batch_pools)
    batch += 1
    print(f"*INFO*: Total pools loaded: {loaded_pools_count}/{len(pool_ids_list)}")

In [0]:
bz_mints_df = spark.createDataFrame(mints_list)
bz_mints_df = bz_mints_df.withColumn("_load_timestamp_bronze", current_timestamp())

### Save & exit

In [0]:
load_result = load_entity(bz_mints_df,
                        entity_name,
                        load_pattern,
                        layer
                        )

In [0]:
dbutils.notebook.exit(load_result)